In [5]:
# 6_cluster.ipynb
#
# Runs K-Means on the normalised feature matrix and produces a
# human-readable DNA table for each cluster persona.
#
# Inputs:
#   normalized.pkl           — Z-scored features (from 5_normalise_ukhls.ipynb)
#   o_indresp_feature_eng.pkl — real values for the DNA report
#
# Outputs:
#   tribe_dna.csv            — human-readable persona profile per cluster
#   pidp_tribe.pkl           — pidp → tribe_id mapping (for step 7)

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_processing.ukhls_variables as _ukhls_vars
importlib.reload(_ukhls_vars)

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

from data_processing.ukhls_variables import (
    VARIABLE_MAP,
    CLUSTER_VARS,
    CATEGORICAL_VARS,
    CATEGORY_MAPS,
)

# ── Config ────────────────────────────────────────────────────────────────────
WAVE            = "o"
NORMALIZED_PKL  = "../data/5_normalise_ukhls/normalized.pkl"
REAL_PKL        = "../data/4_feature_eng_ukhls/o_indresp_feature_eng.pkl"
OUTPUT_CSV      = "../data/6_cluster/tribe_dna.csv"
TRIBE_MAP_PKL   = "../data/6_cluster/pidp_tribe.pkl"
K_CLUSTERS      = 10

# ── Load ──────────────────────────────────────────────────────────────────────
print("Loading data ...")
if not os.path.exists(NORMALIZED_PKL):
    raise FileNotFoundError(f"{NORMALIZED_PKL} not found — run 5_normalise_ukhls.ipynb first.")
if not os.path.exists(REAL_PKL):
    raise FileNotFoundError(f"{REAL_PKL} not found — run 4_feature_eng_ukhls.ipynb first.")

df_norm = pd.read_pickle(NORMALIZED_PKL)
df_real = pd.read_pickle(REAL_PKL)
print(f"Normalised: {df_norm.shape}    Real: {df_real.shape}")

# ── K-Means ───────────────────────────────────────────────────────────────────
print(f"\nRunning K-Means (K={K_CLUSTERS}) ...")
feature_cols = [c for c in df_norm.columns if c != 'pidp']

# Fill any remaining NaNs with 0 (= the mean for Z-scored data)
n_nan_rows = df_norm[feature_cols].isna().any(axis=1).sum()
if n_nan_rows:
    print(f"  Warning: {n_nan_rows:,} rows with NaN in normalised features — filling with 0 (Z-score mean)")
    df_norm[feature_cols] = df_norm[feature_cols].fillna(0)

km = KMeans(n_clusters=K_CLUSTERS, init='k-means++', n_init=20, random_state=42)
df_norm['tribe_id'] = km.fit_predict(df_norm[feature_cols])
print(f"  Inertia: {km.inertia_:,.0f}")

# ── Save pidp → tribe_id mapping ──────────────────────────────────────────────
pidp_tribe = df_norm[['pidp', 'tribe_id']].copy()
pidp_tribe['tribe_id'] = pidp_tribe['tribe_id'].astype('int8')
pidp_tribe.to_pickle(TRIBE_MAP_PKL)
print(f"\nSaved pidp→tribe mapping ({len(pidp_tribe):,} rows) to {TRIBE_MAP_PKL}")

# Join tribe_id onto real values
df_profiling = pd.merge(df_real, df_norm[['pidp', 'tribe_id']], on='pidp')

# ── DNA Table ─────────────────────────────────────────────────────────────────
print("\nBuilding DNA table ...")

def get_mode(series):
    m = series.dropna().mode()
    return float(m.iloc[0]) if not m.empty else np.nan

dna_rows = []
for tribe_id in sorted(df_profiling['tribe_id'].unique()):
    tribe = df_profiling[df_profiling['tribe_id'] == tribe_id]
    row = {'tribe_id': tribe_id, 'size': len(tribe)}

    for base_code in CLUSTER_VARS:
        col = f"{WAVE}_{base_code}"
        if col not in tribe.columns:
            continue
        label = VARIABLE_MAP.get(base_code, base_code)
        series = pd.to_numeric(tribe[col], errors='coerce')

        if base_code in CATEGORICAL_VARS and base_code in CATEGORY_MAPS:
            cat_map  = CATEGORY_MAPS[base_code]
            mode_val = get_mode(series)
            row[label] = cat_map.get(mode_val, str(mode_val))
        else:
            row[label] = round(series.mean(), 1)

    dna_rows.append(row)

dna = pd.DataFrame(dna_rows).sort_values('size', ascending=False).reset_index(drop=True)
dna.to_csv(OUTPUT_CSV, index=False)
print(f"Saved DNA table to {OUTPUT_CSV}")
print(f"\n{dna.to_string(index=False)}")


Loading data ...
Normalised: (47354, 28)    Real: (47354, 28)

Running K-Means (K=10) ...
  Inertia: 824,177

Saved pidp→tribe mapping (47,354 rows) to ../data/6_cluster/pidp_tribe.pkl

Building DNA table ...
Saved DNA table to ../data/6_cluster/tribe_dna.csv

 tribe_id  size  Derived age at interview  Highest qualification  Monthly net pay (take-home)  Social class (NS-SEC 8)  Total monthly personal income (gross)  Household size  Number of children in household  Mental health score (SF-12 MCS)  Physical health score (SF-12 PCS)  Buckner Neighbourhood Cohesion Index  Standard of local services: Public transport  Standard of local services: Shopping  Standard of local services: Leisure  Minutes spent travelling to work  Environmental habit: public transport use  Miles driven in last 12 months English first language (binary) Works at home Drives to work Disability: Mobility Disability: Visual Disability: Hearing Disability: Learning Disability: Mental Health Disability: Manual Dexterity